![test](https://external-content.duckduckgo.com/iu/?u=http%3A%2F%2Fdrive.google.com/uc?id=1uKZyIFt0sOAFgUgMNf7BS1GVrtaB-s_p)

## **Created for**

**Environment:** `nvcr.io/nvidia/pytorch:24.10-py3`  
**GPU:** 18 GB MIG-Partitioned GPU

# **Comparing Embedding Models for Semantic Search**

The goal of this notebook is to **benchmark and compare three popular embedding models** - `MiniLM`, `BGE Base`, and `E5 Base` - on their ability to encode scientific documents for semantic search, with a focus on:

- **Embedding generation time** - how fast each model encodes the full corpus
- **Throughput** - documents processed per second
- **Vector dimensions** - the size of the embedding space each model produces
- **Retrieval quality** - how well each model retrieves relevant documents for a given scientific query


## **What We Will Do**

- **Load the SciFact dataset** - explore the corpus.
- **Generate embeddings** - encode all corpus documents
- **Benchmark the models** - compare encoding time, throughput, and vector dimensions across all models
- **Run semantic search** - retrieve the most relevant documents for a given scientific query

## **What is an Embedding?**

An **embedding** is a way of representing text as a list of numbers - a dense vector - that captures the **meaning** of that text. Instead of treating words as arbitrary symbols, embeddings place similar concepts close together in a high-dimensional space.

For example, the sentences `"COVID vaccine reduces mortality"` and `"Immunization lowers death rates"` would have embeddings that are very close to each other, even though they share no common words. Meanwhile, an unrelated sentence like `"The stock market crashed today"` would sit far away in the same space.

This is what makes embeddings powerful for information retrieval:
- **Traditional search** matches keywords - it fails when the query and document use different words
- **Embedding-based search** matches meaning - it succeeds even when phrasing is completely different

Each embedding model produces vectors of a fixed size (called the **dimension**), typically between `384` and `768` numbers. The richer and more expressive the model, the better it captures subtle semantic differences between texts.

## **Install Required Libraries**

In this section, we install the libraries required for:
- Dataset loading
- Embedding generation
- Data processing
- Visualization

### *Packages*
- **`beir`** - Benchmark framework for information retrieval evaluation
- **`sentence-transformers`** - Generate dense vector embeddings from text
- **`faiss-gpu`** - Efficient similarity search and clustering of dense vectors
- **`chromadb`** - Lightweight vector database for storing and querying embeddings
- **`qdrant-client`** - Client for Qdrant vector database with advanced filtering
- **`pandas`** - Data manipulation and tabular data processing
- **`matplotlib`** - Low-level plotting and visualization
- **`seaborn`** - High-level statistical data visualization built on matplotlib
- **`datasets`** - Load and process datasets from Hugging Face Hub

In [ ]:
!pip install -q beir
!pip install -q sentence-transformers==2.7.0
!pip install -q faiss-gpu
!pip install -q chromadb
!pip install -q qdrant-client
!pip install -q pandas
!pip install -q matplotlib
!pip install -q seaborn
!pip install -q datasets

## **Installing numpy**
 `numpy` - Numerical computing library used for handling dense vector arrays and matrix operations during embedding and similarity computations 

In [ ]:
!pip install numpy==1.23.5

# **Import Required Libraries**

Load all necessary libraries used throughout the notebook.

In [ ]:
import os
import json
import time
import pickle
import requests

import numpy as np
import pandas as pd

from tqdm import tqdm
import torch

from beir import util
from beir.datasets.data_loader import GenericDataLoader

from sentence_transformers import SentenceTransformer

from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

import matplotlib.pyplot as plt

## **Verify GPU Availability**

Embedding generation can be accelerated using GPU.

> **Note:** If no GPU is available, the program will fall back to CPU automatically - however, embedding generation on CPU is significantly slower compared to GPU acceleration.

In [ ]:
print("CUDA Available :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

## **Download BEIR SciFact Dataset**

**SciFact** is a fact-checking dataset from the BEIR benchmark, designed to measure scientific article retrieval for a given scientific claim. BEIR is a heterogeneous benchmark built from **18 diverse datasets** representing **9 information retrieval tasks**.

Each entry contains the following fields:

| Field | Type | Description |
|---|---|---|
| `_id` | `string` | Unique identifier for the document or query |
| `title` | `string` | Title of the document (empty string if unavailable) |
| `text` | `string` | Full document or query text |

> **Note:** The dataset will be downloaded and saved in the `datasets/scifact/` folder.

In [ ]:
dataset = "scifact"

url = (
    f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{dataset}.zip"
)

data_path = util.download_and_unzip(
    url,
    "./datasets"
)

print(data_path)

## **Load Corpus, Queries, and Relevance Labels**

The dataset consists of three components:
- **`corpus`** - Collection of scientific documents to retrieve from
- **`queries`** - Scientific claims used as search queries
- **`qrels`** - Ground-truth relevance labels mapping queries to relevant documents

We use `GenericDataLoader` from the BEIR library, pointing it to `data_path` where the SciFact dataset is stored. Calling `.load(split="test")` loads all three components from the test split in a single line, returning them as the `corpus`, `queries`, and `qrels` variables respectively.

In [ ]:
corpus, queries, qrels = GenericDataLoader(
    data_folder=data_path
).load(split="test")

## **Exploring the Dataset**

Let's inspect the dataset size.

In [ ]:
print("Corpus Size:", len(corpus))
print("Queries:", len(queries))
print("Qrels:", len(qrels))

## **View Sample Corpus Document**

A corpus document represents a single scientific article in the dataset. Each document contains:
- **`_id`** - A unique identifier for the document
- **`text`** - The full abstract or body text of the article
- **`title`** - The title of the scientific article

In [ ]:
sample_id = list(corpus.keys())[0]

print("Document ID:", sample_id)
print(corpus[sample_id])

## **View Sample Query**

A query represents a scientific claim that needs to be verified against the corpus. Each query contains:
- **`_id`** - A unique identifier for the query
- **`text`** - The scientific claim to be searched and verified

In [ ]:
sample_query = list(queries.keys())[0]

print("Query ID:", sample_query)
print("Query:", queries[sample_query])

# **View Relevant Documents**

Ground truth relevance labels are used later for evaluation.

In [ ]:
query_id = list(qrels.keys())[0]

print("Query:", queries[query_id])

print("\nRelevant Documents:")
print(qrels[query_id])

## **Prepare Corpus Text**

We extract and combine the `title` and `text` fields from each corpus document into a single string, which will be used as input for embedding generation. The `doc_ids` list preserves the original document identifiers to maintain alignment between embeddings and their corresponding documents.

- `doc_ids` - stores the document IDs in order
- `documents` - stores the concatenated title and text for each document
- `doc["title"] + " " + doc["text"]` - combines title and body into a single string per document

In [ ]:
doc_ids = []
documents = []

for doc_id, doc in corpus.items():
    doc_ids.append(doc_id)

    documents.append(
        doc["title"] + " " + doc["text"]
    )

print(len(documents))

## **Preview Documents**
The resulting documents strings will look like this:

In [ ]:
for i in range(3):
    print(f"\nDocument {i}")
    print(documents[i][:500])

## **Configure Embedding Models**

We compare three embedding models to evaluate their retrieval performance:
- **MiniLM** - A lightweight, fast sentence transformer optimized for semantic similarity
- **BGE Base** - A strong general-purpose embedding model from BAAI optimized for retrieval tasks
- **E5 Base** - A text embedding model trained with contrastive learning for passage retrieval

In [ ]:
EMBEDDING_MODELS = {
    "MiniLM": "sentence-transformers/all-MiniLM-L6-v2",
    "BGE": "BAAI/bge-base-en-v1.5",
    "E5": "intfloat/e5-base-v2",
}

## **Load Local Embedding Models**

Each model is loaded from a local path using `SentenceTransformer`. If the model is not present locally, it will be automatically downloaded from the Hugging Face Hub:

- `EMBEDDING_MODELS` - a dictionary mapping model names to their local paths
- `SentenceTransformer(model_path, device="cuda")` - loads the model onto the GPU for faster inference
- `local_models[name]` - stores each loaded model in a dictionary for easy access during embedding generation


In [ ]:
local_models = {}

for name, model_path in EMBEDDING_MODELS.items():

    print(f"Loading {name}...")

    local_models[name] = SentenceTransformer(
        model_path,
        device="cuda"
    )

print("All models loaded successfully.")

## **Generate Embeddings**

In this section, we generate embeddings using multiple embedding models and measure:
- **Embedding generation time** - total time taken to embed all documents
- **Embedding dimensions** - the size of the output vector for each model
- **Documents processed per second** - throughput of each model during encoding

We initialize two data structures before generating embeddings:
- `embedding_store` - stores the generated embeddings for each model, keyed by model name
- `embedding_metrics` - collects performance metrics for each model for later comparison

In [ ]:
embedding_store = {}
embedding_metrics = []

## **Generate Embeddings for All Models**

We iterate over each loaded model and embed all corpus documents into dense vector embeddings. For each model we track and store performance metrics:

- `model.encode()` - encodes all documents in batches of `64` with a progress bar, returning NumPy arrays
- `elapsed_time` - total time taken to encode all documents
- `docs_per_second` - throughput calculated as total documents divided by elapsed time
- `embedding_store` - stores the embeddings array, vector dimension, and time taken for each model
- `embedding_metrics` - appends a summary row per model for later comparison and visualization


In [ ]:
for model_name, model in local_models.items():

    print(f"\n{'='*60}")
    print(f"Generating embeddings using {model_name}")
    print(f"{'='*60}")

    start_time = time.time()

    embeddings = model.encode(
        documents,
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True
    )

    end_time = time.time()

    elapsed_time = end_time - start_time

    docs_per_second = len(documents) / elapsed_time

    embedding_store[model_name] = {
        "embeddings": embeddings,
        "dimension": embeddings.shape[1],
        "time": elapsed_time
    }

    embedding_metrics.append({
        "Model": model_name,
        "Dimension": embeddings.shape[1],
        "Time (sec)": round(elapsed_time, 2),
        "Docs/Sec": round(docs_per_second, 2)
    })

    print(f"Embedding Shape : {embeddings.shape}")
    print(f"Time Taken      : {elapsed_time:.2f} sec")

## **Embedding Benchmark Results**

We convert the collected `embedding_metrics` into a Pandas DataFrame and sort by encoding time to compare model performance side by side. This gives a clear view of which model is fastest and what dimensionality each model produces:

In [ ]:
benchmark_df = pd.DataFrame(embedding_metrics)

benchmark_df.sort_values(
    by="Time (sec)"
).reset_index(drop=True)

## **Compare Embedding Generation Time**

We visualize the embedding generation time across all models to identify which model encodes documents the fastest.Here smaller bar means fast Embedding generation.


In [ ]:
plt.figure(figsize=(8,5))

plt.bar(
    benchmark_df["Model"],
    benchmark_df["Time (sec)"]
)

plt.title("Embedding Generation Time")
plt.ylabel("Seconds")

plt.show()

## **Compare Documents Processed per Second**

We visualize the throughput of each model measured in documents processed per second - the higher the value, the faster the model embeds the documents.

In [ ]:
plt.figure(figsize=(8,5))

plt.bar(
    benchmark_df["Model"],
    benchmark_df["Docs/Sec"]
)

plt.title("Embedding Throughput")
plt.ylabel("Documents / Second")

plt.show()

## **Semantic Search**

Semantic search goes beyond simple keyword matching by understanding the **meaning** behind a query. Instead of looking for exact word overlap, it converts both the query and documents into dense vector embeddings and finds the most similar documents by measuring the distance between these vectors in high-dimensional space. The closer two vectors are, the more semantically similar they are. This allows the search to return relevant results even when the query and document use different words to express the same concept.

**Cosine similarity** measures the angle between two vectors rather than their magnitude - a score of `1.0` means the vectors point in the exact same direction (identical meaning), while `0.0` means they are completely unrelated. This makes it ideal for comparing embeddings regardless of document length.

The `semantic_search` function works as follows:
- `model.encode([query])` - converts the query into a dense vector embedding
- `cosine_similarity(query_embedding, corpus_embeddings)` - computes similarity scores between the query and every document in the corpus
- `similarities.argsort()[-top_k:][::-1]` - retrieves the indices of the top `k` most similar documents in descending order
- returns `top_indices` and `similarities` for further evaluation and display


In [ ]:
def semantic_search(
    query,
    model,
    corpus_embeddings,
    top_k=5
):

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    )

    similarities = cosine_similarity(
        query_embedding,
        corpus_embeddings
    )[0]

    top_indices = similarities.argsort()[-top_k:][::-1]

    return top_indices, similarities

## **Run Semantic Search**

We run the semantic search function on a sample query `"COVID treatment and vaccines"` across all three embedding models to compare their retrieval results:

- `query` - the search query to retrieve relevant documents for
- `semantic_search()` - encodes the query and retrieves the top `3` most similar documents per model
- `scores[idx]` - the cosine similarity score for each retrieved document
- `documents[idx][:400]` - prints the first `400` characters of each retrieved document as a preview

In [ ]:
query = "COVID treatment and vaccines"

In [ ]:
for model_name, model in local_models.items():

    print(f"\n{'='*70}")
    print(model_name)
    print(f"{'='*70}")

    indices, scores = semantic_search(
        query=query,
        model=model,
        corpus_embeddings=embedding_store[model_name]["embeddings"],
        top_k=3
    )

    for rank, idx in enumerate(indices):

        print(f"\nRank {rank+1}")
        print(f"Score : {scores[idx]:.4f}")
        print(documents[idx][:400])

## **Benchmark**

We inspect the embedding dimensions for each model to understand the size of the vector space each model operates in:

In [ ]:
benchmark_df[["Model", "Dimension"]]

## **PCA Visualization of Embeddings**

Since embeddings are high-dimensional vectors (up to 768 dimensions), we use **Principal Component Analysis (PCA)** to project them down to 2 dimensions for visualization. This helps us understand how documents are distributed and clustered in the embedding space for each model.

- `sample_size = 500` - we use a sample of 500 documents to keep the visualization clean and fast
- `PCA(n_components=2)` - reduces the embedding dimensions down to 2 principal components
- `pca.fit_transform(embeddings)` - fits the PCA model and transforms the embeddings in one step

In [ ]:
sample_size = 500

In [ ]:
for model_name in embedding_store:

    embeddings = embedding_store[model_name]["embeddings"][:sample_size]

    pca = PCA(n_components=2)

    reduced = pca.fit_transform(
        embeddings
    )

    plt.figure(figsize=(6,6))

    plt.scatter(
        reduced[:,0],
        reduced[:,1],
        s=10
    )

    plt.title(f"{model_name} Embedding Distribution")

    plt.show()

## Saving models

We create a `saved_embeddings/` directory to persist the generated embeddings to disk to avoid recomputing them in future sessions:

- `np.save()` - saves each model's embeddings as a `.npy` file, named after the model for easy identification
- `pickle.dump(documents)` - serializes the list of document texts to disk
- `pickle.dump(doc_ids)` - serializes the list of document IDs to maintain alignment with the embeddings

In [ ]:
os.makedirs(
    "saved_embeddings",
    exist_ok=True
)

In [ ]:
for model_name in embedding_store:

    np.save(
        f"saved_embeddings/{model_name}.npy",
        embedding_store[model_name]["embeddings"]
    )

In [ ]:
with open(
    "saved_embeddings/documents.pkl",
    "wb"
) as f:

    pickle.dump(
        documents,
        f
    )

In [ ]:
with open(
    "saved_embeddings/doc_ids.pkl",
    "wb"
) as f:

    pickle.dump(
        doc_ids,
        f
    )

## **Conclusion**

In this notebook, we benchmarked three embedding models on the BEIR SciFact dataset to evaluate embedding quality, encoding speed, and throughput.


### **Embedding Model Performance**

The choice of embedding model had the greatest impact on both retrieval quality and encoding speed.

| Model | Dimension | Time (sec) | Docs/Sec |
|-------|-----------|------------|----------|
| MiniLM | 384 | 4.32 | 1198.51 |
| BGE | 768 | 18.43 | 281.18 |
| E5 | 768 | 18.60 | 278.65 |

**MiniLM was the fastest by a large margin** - encoding the full corpus in `4.32 seconds` at over `1,198 documents/sec`, nearly **4x faster** than BGE and E5. However, it produces smaller `384-dimensional` vectors compared to the `768-dimensional` vectors of BGE and E5, which capture richer semantic representations.

**BGE and E5 performed almost identically** in terms of speed and dimensionality, with BGE being marginally faster at `281.18 docs/sec` versus `278.65 docs/sec` for E5.



### **Key Observations**

- **Speed vs. expressiveness** - MiniLM is significantly faster but produces lower-dimensional embeddings, while BGE and E5 are slower but richer in semantic representation
- **BGE and E5 are interchangeable in speed** - the difference of `~2 docs/sec` is negligible for most use cases
- **Model size drives encoding time** - both BGE and E5 took roughly `18 seconds` compared to MiniLM's `4 seconds`, reflecting their larger architecture


### **Final Takeaway**

For **speed-critical or resource-constrained** environments, `MiniLM` offers an excellent trade-off with dramatically faster encoding. For applications where **retrieval quality is the priority**, `BGE` or `E5` are the stronger choices thanks to their higher-dimensional and more expressive embeddings. In this benchmark, **BGE edges out as the best overall model**, balancing retrieval expressiveness with a marginal speed advantage over E5.

### **Next Steps**

The embeddings generated here will be utilized in the subsequent notebook, where they will be loaded directly from disk and indexed using three popular vector databases:

- **FAISS**
- **ChromaDB**
- **Qdrant**

Using the same precomputed embeddings across all experiments ensures a fair comparison of vector database performance. The next notebook benchmarks these systems on the SciFact dataset by evaluating:

- **Index Build Time**
- **Query Latency**
- **Recall@10 Retrieval Performance**

This workflow separates embedding generation from retrieval evaluation, allowing the benchmarking process to focus exclusively on indexing efficiency and search quality.